# 1. 학습 데이터 살펴보고 나누기
검수가 끝난 `pairs_checked.csv`를 **살펴보고 → 학습 · 평가 · 시험으로 나누고 → 잘 나뉘었는지 확인**해요.

| 파일 | 역할 | 비유 |
|---|---|---|
| `train.jsonl` | 모델이 보고 배우는 문제와 정답 | 교과서 |
| `eval.jsonl` | 학습 중간에 실력 확인 (15%) | 쪽지시험 |
| `test.jsonl` | 학습이 끝난 뒤 세 모델 비교 (5개) | 기말고사 — **학습에 절대 안 써요** |

준비: `pairs_checked.csv` (칸: `본문`, `리드`, `판정` = 사용 · 수정 · 제외). **수정**한 쌍은 `리드` 칸을 고친 문장으로 바꿔 두세요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

## 설정 — 어떤 데이터인지 고르기
같은 노트북으로 보도자료 데이터와 방송 원고 데이터를 모두 다룰 수 있어요. 지시문(instruction)은 학습 · 비교 내내 **똑같이** 쓰여요.

In [ ]:
# ▶ 설정 — 여기만 고치세요
DATA_FILE = 'pairs_checked.csv'
TASK = '보도자료'                 # '보도자료' 또는 '방송원고'

INSTRUCTIONS = {
    '보도자료': '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.',
    '방송원고': '스포츠 리포트 본문을 읽고, 앵커가 읽을 도입 멘트(리드)를 MBC 뉴스 말투로 쓰세요.',
}
INSTRUCTION = INSTRUCTIONS[TASK]
N_TEST, EVAL_FRAC, SEED = 5, 0.15, 42  # 시험 5개 · 평가 15% · 다시 돌려도 같게 나누는 번호
print('지시문:', INSTRUCTION)

## 1. 데이터 살펴보기

In [ ]:
import pandas as pd
df_all = pd.read_csv(DATA_FILE)
print('전체', len(df_all), '쌍')
print(df_all['판정'].value_counts().to_string())

df = df_all[df_all['판정'] != '제외'].copy()          # 제외한 쌍 빼기
df['본문 글자 수'] = df['본문'].str.len()
df['리드 글자 수'] = df['리드'].str.len()
print('\n학습에 쓸 쌍:', len(df))
df[['본문 글자 수', '리드 글자 수']].describe().round(0)

In [ ]:
# 본문과 리드의 길이 — 모델은 '긴 본문 → 짧은 리드'로 줄여 쓰는 법을 배워요
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(df['본문 글자 수'], bins=30, color='#9aa5b1'); ax[0].set_title('본문 (입력)', loc='left')
ax[1].hist(df['리드 글자 수'], bins=30, color='#e8590c'); ax[1].set_title('리드 (정답)', loc='left')
for a in ax: a.set_xlabel('글자 수')
plt.tight_layout(); plt.show()
print(f"평균: 본문 {df['본문 글자 수'].mean():.0f}자 → 리드 {df['리드 글자 수'].mean():.0f}자. 학습한 모델도 이 길이로 쓰게 돼요.")

In [ ]:
# 한 쌍 직접 읽어 보기 — 숫자를 바꾸면 다른 쌍이 나와요
row = df.sample(1, random_state=1).iloc[0]
print('[본문]\n' + row['본문'][:500] + ('…' if len(row['본문']) > 500 else ''))
print('\n[리드 = 정답]\n' + row['리드'])

## 2. 나누기 → jsonl 파일로 저장
파일 형식은 **alpaca**(instruction · input · output)예요. Unsloth Studio와 2번 노트북이 그대로 읽어요.

In [ ]:
df['instruction'] = INSTRUCTION
df['input'], df['output'] = df['본문'], df['리드']

test  = df.sample(N_TEST, random_state=SEED)           # 기말고사 — 학습에 절대 안 써요
rest  = df.drop(test.index)
eval_ = rest.sample(frac=EVAL_FRAC, random_state=SEED) # 쪽지시험
train = rest.drop(eval_.index)                         # 교과서

for d, name in [(train, 'train.jsonl'), (eval_, 'eval.jsonl'), (test, 'test.jsonl')]:
    d[['instruction', 'input', 'output']].to_json(name, orient='records', lines=True, force_ascii=False)
print(f'train {len(train)} · eval {len(eval_)} · test {len(test)}')

## 3. 잘 나뉘었는지 확인

In [ ]:
import json
# ① 세 파일에 같은 쌍이 겹치지 않아야 해요 (겹치면 시험 문제를 미리 본 셈)
a, b, t = set(train.index), set(eval_.index), set(test.index)
print('겹치는 쌍:', len(a & b) + len(a & t) + len(b & t), '개  (0이어야 해요)')

# ② 학습 스텝 수 미리 계산 — 배치 2 × 누적 4 = 8문제마다 한 번 고치기, 에포크 2
import math
print('예상 학습 스텝:', math.ceil(len(train) / 8) * 2)

# ③ 파일 첫 줄 — Studio에 올릴 때 이 모양이에요
first = json.loads(open('train.jsonl', encoding='utf-8').readline())
print('\ntrain.jsonl 첫 줄')
for k, v in first.items():
    print(f'  {k:12} {v[:60]}…' if len(v) > 60 else f'  {k:12} {v}')

## 다음 단계
1. **학습** — Unsloth Studio(`127.0.0.1:8888`)에 `train.jsonl` · `eval.jsonl`을 올리거나, `2_train_unsloth_gpu.ipynb`
2. **내보내기 · 등록** — GGUF(Q4_K_M)로 내보낸 뒤 `ollama create 내모델이름 -f Modelfile` (README 참고)
3. **비교** — `4_compare_results.ipynb`

⚠ `pairs_checked.csv`와 `*.jsonl`에는 기사 원문이 들어 있어요. GitHub에 올리지 않아요.